In [1]:
# 本部分用于评价LLM的应用程序

In [2]:
import os

from langchain_core.utils import print_text
from langchain_ollama import OllamaEmbeddings, ChatOllama

api_key = os.environ.get("DEEPSEEK_API_KEY")

In [18]:
from langchain_classic.chains import RetrievalQA
from langchain_classic.document_loaders import CSVLoader
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_classic.vectorstores import DocArrayInMemorySearch

In [4]:
file = "OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file, encoding="utf-8") # 注意说明编码
data = loader.load()

In [5]:
# 创建embedding对象
# embeddings = OpenAIEmbeddings(
#     # api_key= os.environ.get("QWEN_API_KEY"), # 填写千问的api-key
#     # # url
#     # base_url= "https://dashscope.aliyuncs.com/compatible-mode/v1",
#     # # 填写目标模型
#     # model = "qwen3.7-text-embedding",
#     # # 发送原始文本
#     # check_embedding_ctx_length=False,
#     # # 根据DashScope的文档，在此处限制单词发送的大小为20
#     # chunk_size=20
#     # 调用本地的Embedding模型
#
# )

embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

# 创建索引
index = VectorstoreIndexCreator(
    # 需要填写embedding参数，未使用OpenAI官方的embedding模型需要手动导入
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings
).from_loaders([loader])

In [19]:
# 设置语言模型
llm = ChatOllama(
    model="qwen3.5:2b",
    temperature=0.0,
    reasoning=False # 关闭思考模式
)


qa = RetrievalQA.from_chain_type(
    llm=llm,
    # 设置链的类型
    chain_type="stuff",
    # 设置检索器
    retriever=index.vectorstore.as_retriever(),
    verbose=True, # 设置打印日志详细程度
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>"
    }
)

In [7]:
# 自身预设一些设置好的数据集
data[10]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [8]:
data[11]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

In [9]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

In [10]:
# 导入QA生成链
from langchain_classic.evaluation.qa import QAGenerateChain

In [11]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/",
    model="deepseek-v4-flash",
    temperature=0.0
)
)

In [12]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

C:\Users\hilary\AppData\Local\Temp\ipykernel_12220\3125874183.py:1: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(
F:\LangChain\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:564: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")


In [13]:
new_examples[0] # 可以看到在当前的example里面，前面包裹了一个qa_pairs字段，需要先对其进行处理

{'qa_pairs': {'query': "According to the product description, what are the size guidance, approximate weight, and key construction features of the Women's Campside Oxfords?",
  'answer': 'Order regular shoe size; for half sizes not offered, order up to the next whole size. The approximate weight is 1 lb. 1 oz. per pair. Key construction features include a soft canvas material, a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, a vintage hunt/fish/camping motif on the innersole, a moderate arch contour, an EVA foam midsole, and a chain-tread-inspired molded rubber outsole with a modified chain-tread pattern.'}}

In [14]:
data[0]

Document(metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [15]:
examples += [e["qa_pairs"] for e in new_examples]


In [16]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "According to the product description, what are the size guidance, approximate weight, and key construction features of the Women's Campside Oxfords?",
  'answer': 'Order regular shoe size; for half sizes not offered, order up to the next whole size. The approximate weight is 1 lb. 1 oz. per pair. Key construction features include a soft canvas material, a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, a vintage hunt/fish/camping motif on the innersole, a moderate arch contour, an EVA foam midsole, and a chain-tread-inspired molded rubber outsole with a modified chain-tread pattern.'},
 {'query': 'What are the dimensions for the Medium size of the Recycled Waterhog Dog Mat, and what materials is it made from?',
  'answer': 'The Medium size 

In [20]:
# 将某个示例传入链并且运行
qa.run(examples[0]["query"]) # 此处无法查看传给语言模型的Prompt是什么



> Entering new RetrievalQA chain...

> Finished chain.


'Yes, the Cozy Comfort Pullover Set has side pockets. According to the description, the pull-on pants feature a wide elastic waistband and drawstring, along with **side pockets**.'

In [21]:
# 对传递chain的每一个步骤进行观察
from langchain_core.globals import set_debug

set_debug(True)


In [22]:
qa.run(examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set have side pockets?",
  "context": ": 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. \n\nSize & Fit \nPants are Favorite Fit: Sits lower on the waist. \nRelaxed Fit: Our most generous fit sits farthest from the body. \n\nFabric & Care \nIn the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features \nRelaxed fit top with raglan sleeves and rounded hem. \nPull-on pants have a wide elastic

'Yes, the Cozy Comfort Pullover Set has side pockets. According to the description, the pull-on pants feature a wide elastic waistband and drawstring, along with **side pockets**.'

In [23]:
# 输入前文创建的所有示例
set_debug(False)

In [24]:
predictions = qa.apply(examples)

C:\Users\hilary\AppData\Local\Temp\ipykernel_12220\1205324748.py:1: LangChainDeprecationWarning: The method `Chain.apply` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `batch` instead.
  predictions = qa.apply(examples)




> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


In [25]:
# 对上述示例进行评估
from langchain_classic.evaluation.qa import QAEvalChain

In [34]:
# 使用语言模型创建该chain
llm = ChatOllama(
    model="qwen3.5:2b",
    temperature=0.0,
    reasoning=False # 关闭思考模式
)
# 使用语言模型帮助评估
eval_chain = QAEvalChain.from_llm(llm)

In [42]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [56]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question:" + predictions[i]['query'])
    print("Real Answer:" + predictions[i]["answer"])
    print("Predicted Answer:" + predictions[i]['result'])
    print("Predicted Grade:" + graded_outputs[i]['results'][6:])
    print()

# 根据上述结果可知，无法直接使用简单的正则表达式进行评估，因此需要借助语言模型，来对不唯一的答案进行评估

Example 0:
Question:Do the Cozy Comfort Pullover Set have side pockets?
Real Answer:Yes
Predicted Answer:Yes, the Cozy Comfort Pullover Set has side pockets. According to the description, the pull-on pants feature a wide elastic waistband and drawstring, along with **side pockets**.
Predicted Grade: CORRECT

Example 1:
Question:What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real Answer:The DownTek collection
Predicted Answer:Based on the context provided, the **Ultra-Lofty 850 Stretch Down Hooded Jacket** is from the **DownTek collection**.

The description explicitly states: "This technical stretch down jacket from our **DownTek collection**..."
Predicted Grade: CORRECT

Example 2:
Question:According to the product description, what are the size guidance, approximate weight, and key construction features of the Women's Campside Oxfords?
Real Answer:Order regular shoe size; for half sizes not offered, order up to the next whole size. The approximate weight is 1